# Coin Counting Machine

![coin_counter_image.png](https://live.staticflickr.com/65535/54326911769_4e446cdd97_k.jpg)
*Image generated using the Flux-dev model.*

## Introduction
Imagine that you discover a large chest full of coins in the attic and decide to count their total value. Manually counting each coin would be time-consuming and boring, so you decide to automate the solution. Based on images of scattered coins, your goal is to determine their total value.
By breaking this task down into its basic components, your task will be to:
* locate each coin in the image,
* assign them to appropriate categories according to denominations (e.g. 1 grosz or 2 złote).

This task in the field of computer vision is called object detection. Note that detection is a simultaneous task of both classification and regression — you must determine both the class of the object and its location in the image. An additional challenge that distinguishes detection from a standard classification task is the fact that instead of a single object representing one of the classes, an image may contain many objects of different classes.

Formally, we can say that the solution $\hat{\mathcal{B}}$ should be a set of tuples containing, respectively, the coordinates of the rectangle in which the coin is located, the label specifying the coin’s denomination, and the confidence with which the model believes it has found a coin.
$$
\hat{\mathcal{B}} = \{(x_{min}, y_{min}, x_{max}, y_{max}, c, \phi), \dots\}
$$
where $(x_{min}, y_{min})$ is the top-left corner of the rectangle, $(x_{max}, y_{max})$ is the bottom-right corner of the rectangle, $c$ is the coin label, and $\phi$ is the model’s confidence in the prediction. Depending on the image, the set $\hat{\mathcal{B}}$ may contain a varying number of elements, corresponding to the number of detected coins. The order of coins in the set does not matter.

*Note*: When working with rectangle positions in the image, note that the point (0, 0) is located in the top-left corner of the image, and the Y axis increases downward. This is the standard coordinate system used in computer graphics, which differs from the coordinate system used in mathematics.

During detection, we can make five types of errors, which are illustrated in the images below:
- The model detected an object where there is none (this situation is called a False Positive)
- The model did not detect an object that is present in the image (this situation is called a False Negative)
- The model detected an object but classified it incorrectly
- The model correctly detected and classified an object but returned imprecise rectangle coordinates
- The model detected the same object multiple times

![coin_counter_prediction.png](https://live.staticflickr.com/65535/54327101055_460ce3640e_k.jpg)

In this task, we consider only Polish coins and restrict ourselves to the denomination side. Both banknotes and the side with the eagle image are not included in the dataset and are not taken into account.

[Link](https://github.com/OlimpiadaAI/szkolenia/blob/main/12_Zadania_detekcji_i_segmentacji.pdf) to lecture slides on object detection and segmentation.

## Task
Your task is to implement the class ```YourDetector```, in which the ```forward``` method takes an image and returns a set containing coin predictions in the format described in the definition of the set $\hat{\mathcal{B}}$.

### Evaluation Criterion
*Note*: Both the methods for computing metrics and data loading have been implemented in the task. Your only goal is to implement the detection model, however we encourage you to familiarize yourself with the description of the metrics to better understand the problem.

The task will be evaluated based on the [mAP](https://kili-technology.com/data-labeling/machine-learning/mean-average-precision-map-a-complete-guide) metric (mean Average Precision), which is a standard metric in the field of object detection.  
The computation of this metric begins with matching predictions to ground-truth objects. For this purpose, the IoU metric (Intersection over Union) is used, which determines the degree of overlap between two rectangles. The IoU value is defined as the ratio of the area common to both rectangles $A_{\text{inter}}$ to the area of their union $A_{\text{union}}$.
$$
IoU = \frac{A_{\text{inter}}}{A_{\text{union}}}
$$
Looking at the figure below, we can say that IoU is the ratio of the yellow area to the blue area. It is worth noting that if the rectangles had no intersection, the IoU value would be $0$, and if both rectangles were identical, the IoU value would be $1$.
The IoU value must exceed a fixed threshold for us to consider two rectangles as overlapping. The higher the threshold, the more accurate the object locations returned by the model must be in order to be matched with rectangles created based on the true coordinates.

![coin_counter_explanation_1.png](https://live.staticflickr.com/65535/54326929064_3927b29b3b_k.jpg)

After matching the model predictions to the ground-truth objects, we can compute two key measures: precision and recall. Both measures are computed separately for each class.
- Precision expresses the proportion of objects correctly classified into a given class among all objects that were assigned to that class.
- Recall describes the proportion of correctly classified objects of a given class relative to all objects of that specific class.

These two measures help assess the effectiveness of the model. However, because models are imperfect and make various types of errors, improving one metric usually leads to a deterioration of the other. For example, if we lower the model confidence threshold, we will consider a larger number of objects indicated by the model, but this may simultaneously increase the number of objects falsely indicated by the model as relevant.

To better understand this trade-off, we use the model confidence values (denoted as $\phi$) and determine a precision-recall curve based on them. This curve shows how precision and recall change depending on the chosen confidence threshold.

The area under the curve drawn below is the average precision (AP) for a given class.

![coin_counter_explanation_2.png](https://live.staticflickr.com/65535/54326928103_12e06df704_z.jpg)

The mAP metric is the mean average precision over all classes, i.e. the average area under the precision-recall curve for all classes, described by the formula:
$$
mAP = \frac{1}{K} \sum_{k=1}^K {AP}_{k},
$$
where $K$ is the number of classes and ${AP}_{k}$ is the average precision for class $k$.

The final step is choosing the IoU value for which we want to compute mAP. A standard practice, which we will use to evaluate your solution, is to compute mAP for different IoU values (starting from 0.5 and increasing by 0.05 up to 0.95) and average the results. This way, the metric better reflects not only the quality of classification but also the quality of object localization (for an IoU threshold of 0.95 the model must return predictions that almost perfectly overlap with the target rectangles, whereas for IoU = 0.5 the metric “forgives” much larger differences in position).

**Ultimately, your solution will be evaluated on a hidden test set based on the mAP metric.** The test set does not differ significantly from the validation set.

- If the mAP value for your model is 0.2 (or below), you will receive 0 points for the task.
- If the mAP value for your model is 0.85 (or above), you will receive 100 points for the task.
- Otherwise, the number of points will be determined proportionally to the mAP value:
$$
\text{score} = \frac{mAP - 0.2}{0.85 - 0.2} \times 100
$$

## Constraints
- Your solution will be tested on the Competition Platform without internet access and in an environment with a GPU.
- Evaluation of your final solution on the Competition Platform must not take longer than 10 minutes with a GPU, while evaluation for a single image must not take longer than 5 seconds.
- The model may not use other datasets or pre-trained weights from other datasets.
- The model must return results in a format compatible with the ```predict_all_bounding_boxes``` function (similar to the example solution).
- The model must inherit from the ```nn.Module``` class.

## Submission Files
This notebook supplemented with your solution (see the `YourDetector` class).

## Evaluation
Remember that during evaluation the `FINAL_EVALUATION_MODE` flag will be set to `True`.

For this task, you can score between 0 and 100 points. The number of points you obtain will be calculated on the (hidden) test set on the Competition Platform based on the formula above and rounded to an integer. If your solution does not meet the above criteria or does not execute correctly, you will receive 0 points for the task.

# Starter Code
In this section, we initialize the environment by importing the required libraries and functions. The provided code will help you efficiently work with the data and build the appropriate solution.

In [ ]:
FINAL_EVALUATION_MODE = False

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

import os
import torch
import pickle
import gdown
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import torchvision.transforms.v2 as T
from collections.abc import Callable
from matplotlib import patches
from matplotlib.collections import PatchCollection
from torchvision.models import resnet18
from torchvision.ops import box_iou
from torch.utils.data import Dataset
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert torch.cuda.is_available(), "CUDA niedostępna!"

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

seed = 12345

os.environ["PYTHONHASHSEED"] = str(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
# Komórka zawierająca funkcje pomocnicze do przygotowania danych.


class CoinsDataset(Dataset):
    """
    Zbiór danych monet wczytywany z pliku pickle.

    Przyjmuje:
        pickle_file (str): Ścieżka do pliku pickle zawierającego dane.
        transform (callable, opcjonalnie): Transformacje stosowane do obrazów i etykiet.
    """

    def __init__(self, pickle_file: str, transform: Callable | None = None):
        self.transform = transform

        with open(pickle_file, "rb") as f:
            self.data = pickle.load(f)

    def __len__(self) -> int:
        """Zwraca liczbę próbek w zbiorze danych."""
        return len(self.data)

    def __getitem__(self, idx: int) -> dict:
        """
        Pobiera próbkę danych na podstawie indeksu.

        Przyjmuje:
            idx (int): Indeks próbki.

        Zwraca:
            dict: Słownik zawierający obraz oraz odpowiadające mu obiekty docelowe (boxes, labels).
        """
        sample = self.data[idx]
        image = sample["image"]
        target = {"boxes": sample["boxes"], "labels": sample["labels"]}

        if self.transform:
            image, target = self.transform(image, target)

        return {"image": image, **target}


def setup_data(
    train_transform: Callable | None = None,
    val_transform: Callable | None = None,
    root: str = "data/",
) -> tuple:
    """
    Przygotowuje zbiory danych do trenowania i walidacji, pobierając je jeśli to konieczne.

    Przyjmuje:
        train_transform (callable, opcjonalnie): Augmentacje dla zbioru treningowego.
        val_transform (callable, opcjonalnie): Augmentacje dla zbioru walidacyjnego.
        root (str, opcjonalnie): Katalog bazowy dla plików z danymi.

    Zwraca:
        tuple: Zbiory danych (train_ds, val_ds).
    """
    if train_transform is None:
        train_transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
    if val_transform is None:
        val_transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

    train_file = root + "train.pkl"
    val_file = root + "val.pkl"

    if not os.path.exists(root):
        os.makedirs(root)

    if not os.path.exists(train_file):
        url = "https://drive.google.com/uc?id=1KC8FBlCuwh9ITUt0CcPeRJCpqy5j4WBp"
        gdown.download(url, train_file, quiet=True)

    if not os.path.exists(val_file):
        url = "https://drive.google.com/uc?id=1Oza4UjnmAUeae2cA8YDwMWxVHOb7SKdP"
        gdown.download(url, val_file, quiet=True)

    train_ds = CoinsDataset(root + "train.pkl", transform=train_transform)
    val_ds = CoinsDataset(root + "val.pkl", transform=val_transform)

    return train_ds, val_ds

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
# Komórka zawierająca funkcje pomocnicze do wyznaczenia metryk oceniających jakość modelu.


def predict_all_bounding_boxes(model: nn.Module, ds: Dataset) -> list:
    """
    Funkcja przewidująca wszystkie bounding boxy dla zbioru danych z wykorzystaniem modelu.

    Przyjmuje:
        model: Model detekcji obiektów.
        ds: Zbiór danych.

    Zwraca:
        Lista zawierająca wszystkie przewidziane bounding boxy dla każdej próbki w zbiorze danych.
    """
    all_pred_bboxes = []

    for sample in ds:
        img = sample["image"].to(DEVICE)
        pred_bboxes = model(img)
        all_pred_bboxes.append(pred_bboxes)

    return all_pred_bboxes


def calculate_map(
    predictions: list, ds: Dataset, return_all: bool = False
) -> dict | float:
    """
    Funkcja obliczająca średnią precyzję średnią (mAP) dla przewidzianych bounding boxów dla całego zbioru danych.

    Przyjmuje:
        predictions: Lista zawierająca przewidziane bounding boxy dla każdej próbki w zbiorze danych.
        ds: Zbiór danych, zawierający prawdziwe bounding boxy.
        return_all (bool, opcjonalnie): Czy zwrócić wszystkie metryki, czy tylko mAP:0.5:0.95:0.05.

    Zwraca:
        Wartość mAP lub wszystkie metryki w formie słownika.
    """
    meta = []

    for img_meta in predictions:
        entry = {"boxes": [], "labels": [], "scores": []}

        for box in img_meta:
            entry["boxes"].append(box[:4])
            entry["labels"].append(box[4])
            entry["scores"].append(box[5])

        meta.append(entry)

    for i in range(len(meta)):
        meta[i]["boxes"] = torch.tensor(meta[i]["boxes"])
        meta[i]["labels"] = torch.tensor(meta[i]["labels"]).view(-1)
        meta[i]["scores"] = torch.tensor(meta[i]["scores"])

    mAP = MeanAveragePrecision()

    GT = [{"boxes": sample["boxes"], "labels": sample["labels"]} for sample in ds]

    output = mAP(meta, GT)

    if return_all:
        return output

    return mAP(meta, GT)["map"].item()


def compute_confusion_matrix(
    predictions: list, ds: Dataset, iou_threshold: float = 0.5
) -> np.ndarray:
    """
    Funkcja obliczająca macierz pomyłek dla przewidzianych bounding boxów dla całego zbioru danych.

    Przyjmuje:
        predictions: Lista zawierająca przewidziane bounding boxy dla każdej próbki w zbiorze danych.
        ds: Zbiór danych, zawierający prawdziwe bounding boxy.
        iou_threshold (float, opcjonalnie): Próg IoU dla przypisania predykcji do obiektu.

    Zwraca:
        Macierz pomyłek.
    """
    num_classes = 10  # 9 klas monet + 1 tło
    conf_matrix = np.zeros((num_classes, num_classes), dtype=int)

    for pred_boxes, item in zip(predictions, ds):
        # Ground truth
        gt_boxes = item["boxes"]
        gt_labels = item["labels"]

        # predykcje
        pred_boxes_tensor = (
            torch.tensor([p[:4] for p in pred_boxes])
            if pred_boxes
            else torch.empty((0, 4))
        )
        pred_labels = (
            torch.tensor([p[4] for p in pred_boxes])
            if pred_boxes
            else torch.empty((0,), dtype=torch.long)
        )

        # wartości IoU dla wszystkich par predykcji i ground truth
        iou_matrix = (
            box_iou(pred_boxes_tensor, gt_boxes)
            if pred_boxes
            else torch.empty((0, gt_boxes.shape[0]))
        )

        # na podstawie IoU przypisz predykcje do ground truth
        matched_gt = set()
        for pred_idx, ious in enumerate(iou_matrix):
            max_iou, gt_idx = torch.max(ious, dim=0)
            if max_iou >= iou_threshold and gt_idx.item() not in matched_gt:
                conf_matrix[gt_labels[gt_idx].item(), pred_labels[pred_idx].item()] += 1
                matched_gt.add(gt_idx.item())
            else:
                conf_matrix[-1, pred_labels[pred_idx].item()] += 1  # False positive

        # Dla wszystkich nieprzypisanych obiektów ground truth dodaj jako False negative
        for gt_idx in range(len(gt_boxes)):
            if gt_idx not in matched_gt:
                conf_matrix[gt_labels[gt_idx].item(), -1] += 1

    return conf_matrix

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################
# Komórka zawierająca funkcje pomocnicze do wizualiacji wyników

colors = [
    "red",
    "green",
    "blue",
    "yellow",
    "black",
    "purple",
    "orange",
    "brown",
    "pink",
]
label_names = [
    "1 grosz",
    "2 grosze",
    "5 groszy",
    "10 groszy",
    "20 groszy",
    "50 groszy",
    "1 złotych",
    "2 złote",
    "5 złotych",
]


def show_sample(sample: dict):
    """
    Funkcja wyświetlająca obraz z bounding boxami obiektów.

    Przyjmuje:
        sample: Słownik zawierający obraz oraz etykiety.
    """
    image = sample["image"]
    meta = sample

    plt.figure(figsize=(9, 5))
    plt.imshow(image.permute((1, 2, 0)))
    plt.xticks([])
    plt.yticks([])

    if meta is not None:
        patches_list = []
        legend_labels = []

        for bbox, label in zip(meta["boxes"], meta["labels"]):
            points = np.array(bbox)
            points = points.astype(int)

            # Rysowanie prostokąta, który otacza obiekt
            rect = patches.Rectangle(
                (points[0], points[1]),
                points[2] - points[0],
                points[3] - points[1],
                linewidth=2,
                edgecolor=colors[label.item()],
                facecolor="none",
            )
            patches_list.append(rect)

            # legenda z unikalnymi etykietami
            if label_names[label.item()] not in legend_labels:
                legend_labels.append(label_names[label.item()])

        patch_collection = PatchCollection(patches_list, match_original=True)
        plt.gca().add_collection(patch_collection)

        # Dodanie legendy z unikalnymi etykietami
        handles = [
            patches.Patch(color=colors[i], label=label)
            for i, label in enumerate(label_names)
            if label in legend_labels
        ]
        plt.legend(handles=handles, loc="upper right")

    plt.show()


def plot_detection_results_grid(predictions: list, ds: Dataset, size: tuple = (2, 3)):
    """
    Funkcja wyświetlająca wyniki detekcji na wybranych zdjęciach walidacyjnych.

    Przyjmuje:
        predictions: Lista zawierająca przewidziane bounding boxy dla każdej próbki w zbiorze danych.
        ds: Zbiór danych.
        size (tuple, opcjonalnie): Rozmiar siatki obrazów.
    """

    fig, axes = plt.subplots(size[0], size[1], figsize=(size[1] * 6, size[0] * 5))

    for i, (sample, pred_meta) in enumerate(zip(ds, predictions)):
        if i >= size[0] * size[1]:
            break

        img = sample["image"]

        ax = axes[i // size[1], i % size[1]]
        ax.imshow(img.permute(1, 2, 0))
        ax.axis("off")

        for x1, y1, x2, y2, label, _ in pred_meta:
            rect = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor=colors[label],
                facecolor="none",
            )
            ax.add_patch(rect)

    handles = [
        patches.Patch(color=colors[i], label=label)
        for i, label in enumerate(label_names)
    ]
    plt.legend(handles=handles, loc="upper right")
    plt.suptitle("Detekcja monet na wybranych sześciu zdjęciach walidacyjych")
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(predictions: list, ds: Dataset, iou_threshold: float = 0.5):
    """
    Funkcja wyświetlająca macierz pomyłek dla detekcji monet.

    Przyjmuje:
        predictions: Wynik detekcji monet.
        ds: Zbiór danych.
        iou_threshold float: Próg IoU dla przypisania predykcji do obiektu
    """
    conf_matrix = compute_confusion_matrix(predictions, ds, iou_threshold=iou_threshold)

    labels = [
        "1 grosz",
        "2 grosze",
        "5 groszy",
        "10 groszy",
        "20 groszy",
        "50 groszy",
        "1 złotych",
        "2 złote",
        "5 złotych",
        "brak (tło)",
    ]
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        conf_matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        cbar=False,
    )

    plt.xlabel("Model predykował", labelpad=15)
    plt.ylabel("podczas, gdy powinien był predykować", labelpad=15)
    plt.xticks(rotation=45)

    plt.title(
        "Macierz pomyłek dla detekcji monet z użyciem IOU={}".format(iou_threshold)
    )
    plt.show()

## Data Loading
Using the code below, the data will be loaded and properly prepared.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

train_ds, val_ds = setup_data(root="./data/")

if not FINAL_EVALUATION_MODE:
    print(
        "Ilość zdjęć w zbiorze treningowym:",
        len(train_ds),
        ", ilość zdjęć w zbiorze walidacyjnym:",
        len(val_ds),
    )

    sample = train_ds[0]

    print("Każda próbka zawiera:", list(sample.keys()))
    print("Każde zdjęcie ma wymiary:", list(sample["image"].shape))
    print("Ta próbka posiada", sample["labels"].shape[0], "obiektów")
    print(
        "Każdy prostokąt jest opisany jako",
        sample["boxes"].shape[1],
        "wartości (x1, y1, x2, y2)",
    )

    show_sample(sample)  # wyświetlenie przykładowego zdjęcia z zaznaczonymi obiektami

### Object Classes and Their Labels
Below is a table with the class labels present in the dataset along with their brief descriptions.

| Label | Description |
| --- | --- |
| 0 | Coin with a denomination of 1 grosz |
| 1 | Coin with a denomination of 2 grosze |
| 2 | Coin with a denomination of 5 grosze |
| 3 | Coin with a denomination of 10 grosze |
| 4 | Coin with a denomination of 20 grosze |
| 5 | Coin with a denomination of 50 grosze |
| 6 | Coin with a denomination of 1 złoty |
| 7 | Coin with a denomination of 2 złote |
| 8 | Coin with a denomination of 5 złotych |

## Example Solution
Below we present a simplified solution that serves as an example demonstrating the basic functionality of the notebook. It can be used as a starting point for developing your solution.

As a simple example, a convolutional network applied to a sliding window can be used. This method consists of extracting image patches (in our case, of sizes typical for coins in the dataset) and classifying each of them using a convolutional network. The network outputs one of 10 classes: nine of them correspond to different denominations, while the last class is reserved for the background, i.e. areas where there is no coin. The classification operation is performed for each image patch by sliding the window by a fixed step. If the network returns the background class, we ignore this patch; otherwise, we assign a coin label to the analyzed patch. In this way, we obtain a set of rectangles that contain coins.

When implementing this approach, an additional class representing the background (a rectangle in which there is no coin) must be taken into account. During the training process, we can take random crops of the image. If a crop overlaps with a coin with an IoU value greater than $0.5$, we choose its denomination as the label. If the crop does not overlap with any coin, we choose the background as the label.

We will start by defining a simple model, and in the following cells we will focus on implementing the training function.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################


class BasicCNNClassifier(nn.Module):
    def __init__(self):
        super(BasicCNNClassifier, self).__init__()
        self.net = resnet18(weights=None, num_classes=10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Funkcja klasyfikująca obraz x do jednej z 10 klas: 9 monet + tło.

        Przyjmuje:
            x (torch.Tensor): Obraz do klasyfikacji.

        Zwraca:
            torch.Tensor: Predykcje modelu.
        """
        return self.net(x)

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################
# code for converting an image with multiple coins into training data
# for a classifier
# in 90% of cases we sample an object from the image and crop its surroundings
# in 10% of cases we choose a completely random image fragment


class ClassificationDataPreprocessor:
    def __init__(self, num_of_crops_per_image: int = 4, box_size: int = 64):
        self.num_of_crops_per_image = num_of_crops_per_image
        self.box_size = box_size

    def get_label_of_crop(
        self,
        crop_box: tuple,
        boxes: torch.Tensor,
        labels: torch.Tensor,
        iou_threshold: float = 0.5,
    ) -> int:
        """
        Function returning the label for an image crop depending on whether it
        overlaps with a coin with IoU value > 0.5.

        Takes:
            crop_box (tuple): Coordinates of the top-left and bottom-right corner
                              of the image crop (x1, y1, x2, y2).
            boxes (torch.Tensor): Tensor containing coin bounding boxes.
            labels (torch.Tensor): Tensor containing coin labels.
            iou_threshold (float): IoU threshold for assigning a prediction to an object.
        """
        for box, label in zip(boxes, labels):
            if (
                box_iou(torch.tensor(crop_box).view(1, 4), box.view(1, 4))
                > iou_threshold
            ):
                return label
        return 9  # values 0-8 are coin labels, so 9 will be the background

    def __call__(self, batch: list) -> dict:
        """
        Function processing images with multiple coins into training data
        for a classifier, containing image crops and their labels.

        Takes:
            batch (list): List of dictionaries containing images,
                          coin bounding boxes, and their labels.

        Returns:
            dict: Dictionary containing image crops and their labels.
        """
        crops, labels = [], []

        for sample in batch:
            img = sample["image"]

            for _ in range(self.num_of_crops_per_image):
                if torch.rand(1) < 0.1:  # in 1 out of 10 cases we sample background
                    # (we try to balance the dataset this way)
                    central_x = torch.randint(0, img.shape[2], (1,)).item()
                    central_y = torch.randint(0, img.shape[1], (1,)).item()
                else:  # in the remaining 90% we sample an object from the image
                    # and crop its surroundings
                    idx_gt = torch.randint(0, sample["labels"].shape[0], (1,)).item()
                    central_x = (
                        sample["boxes"][idx_gt, 0] + sample["boxes"][idx_gt, 2]
                    ) // 2 + torch.randint(-10, 10, (1,)).item()
                    central_y = (
                        sample["boxes"][idx_gt, 1] + sample["boxes"][idx_gt, 3]
                    ) // 2 + torch.randint(-10, 10, (1,)).item()

                x1 = np.clip(
                    central_x - self.box_size // 2, 0, img.shape[2] - self.box_size
                )
                y1 = np.clip(
                    central_y - self.box_size // 2, 0, img.shape[1] - self.box_size
                )
                x2, y2 = x1 + self.box_size, y1 + self.box_size

                crop = img[:, y1:y2, x1:x2]
                label = self.get_label_of_crop(
                    (x1, y1, x2, y2), sample["boxes"], sample["labels"]
                )

                crops.append(crop)
                labels.append(label)

        return {"crops": torch.stack(crops, dim=0), "labels": torch.tensor(labels)}

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################
# code for validation using the sliding window method


class SlidingWindowDetector(nn.Module):
    """
    Class implementing an object detector using the sliding window method.
    This is an example solution to the coin detection problem.

    Takes:
        classifier (nn.Module): Object classifier model. For each sliding window
                    position, the classifier returns a prediction for the given
                    image crop.
        crop_size: Size of the sliding window
        stride: distance by which we move the window in each iteration.
    """

    def __init__(self, classifier: nn.Module, crop_size: int = 64, stride: int = 32):
        super(SlidingWindowDetector, self).__init__()
        self.classifier = classifier
        self.crop_size = crop_size
        self.stride = stride

    def forward(self, image: torch.Tensor) -> list:
        """
        Function predicting objects in an image using the sliding window method.

        Takes:
            image: Image to be processed.

        Returns:
            list of detected objects in the form of tuples
            (x1, y1, x2, y2, label, confidence).
        """
        # move the image to the appropriate device (same as the model)
        device = next(self.parameters()).device
        image = image.to(device)

        found_objects = []  # list of objects, where an object is a tuple
        # (x1, y1, x2, y2, label, confidence)

        # slide the window over the image
        for y in range(0, image.shape[1] - self.crop_size, self.stride):
            for x in range(0, image.shape[2] - self.crop_size, self.stride):
                crop = image[:, y : y + self.crop_size, x : x + self.crop_size]
                pred = self.classifier(crop.unsqueeze(0))[0]

                if pred.argmax() != 9:  # ignore background
                    label = pred.argmax().item()
                    confidence = torch.softmax(pred, dim=0)[label].item()
                    found_objects.append(
                        (
                            x,
                            y,
                            x + self.crop_size,
                            y + self.crop_size,
                            label,
                            confidence,
                        )
                    )

        return found_objects

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################


def train_basic_detector(train_ds: Dataset) -> SlidingWindowDetector:
    """
    Function that trains a coin classifier using image crops and classifying them.
    Once the classifier is trained, we create an object detector using the sliding
    window method (SlidingWindowDetector).

    Takes:
        train_ds (Dataset): Training dataset.

    Returns:
        SlidingWindowDetector: Trained object detector using the sliding window method.
    """

    # what should be the size of the sliding window? we can find the answer in the data...
    # let's compute the average size of all objects in the training set
    total = 0
    boxes = 0

    for sample in train_ds:
        total += sum(
            sample["boxes"][:, 2] - sample["boxes"][:, 0]
        )  # x2 - x1 (we assume objects are square, so we ignore y)
        boxes += len(sample["boxes"])

    box_size = (total / boxes).item()

    print("Average object size in the training set:", box_size)  # should be around 62

    # this value is close to 64, which is a power of 2, making the network architecture easier
    box_size = 64
    preprocess = ClassificationDataPreprocessor(
        num_of_crops_per_image=128, box_size=box_size
    )

    # prepare the model, optimizer, and loss function
    # (thanks to using the sliding window technique, our learning problem
    # has become a classification problem)
    model = BasicCNNClassifier()
    model.to(DEVICE)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=0.001
    )
    criterion = nn.CrossEntropyLoss()

    # prepare dataloaders
    train_dl = torch.utils.data.DataLoader(
        train_ds, batch_size=2, shuffle=True, collate_fn=preprocess
    )

    # train the model for a selected number of epochs
    epochs = 30

    pbar = tqdm(range(epochs), desc="Training", total=epochs)
    for _ in pbar:
        epoch_losses = []

        for batch in train_dl:
            X = batch["crops"].to(DEVICE)
            y = batch["labels"].to(DEVICE)

            optimizer.zero_grad()
            output = model(X)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.detach().cpu().item())

        avg_loss = np.mean(epoch_losses)
        pbar.set_postfix({"train loss": avg_loss})

    model.eval()

    # construct the object detector using the sliding window method,
    # which uses the trained classifier to predict crops
    return SlidingWindowDetector(model)

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

if not FINAL_EVALUATION_MODE:
    train_ds, val_ds = setup_data(root="data/")

    model = train_basic_detector(train_ds)

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

if not FINAL_EVALUATION_MODE:
    out = predict_all_bounding_boxes(model, val_ds)

    map_val = calculate_map(out, val_ds, return_all=True)

    print(
        f"mAP z użyciem progu IoU=0.5 na zbiorze walidacyjnym: {map_val['map_50'].item():.2f}"
    )
    print(
        f"mAP dla wielu wartości IoU na zbiorze walidacyjnym:  {map_val['map']:.2f}, to jest metryka, która podlega ocenie w konkursie. Twoim zadaniem jest ją maksymalizować."
    )

    plot_detection_results_grid(out, val_ds)

    plot_confusion_matrix(out, val_ds, iou_threshold=0.5)
    plot_confusion_matrix(out, val_ds, iou_threshold=0.8)

It is worth paying attention to the discrepancies in mAP values when using an IoU threshold of 0.5 versus using many different IoU values (0.5, ..., 0.95). A similar discrepancy is visible in the confusion matrix for different IoU values. This means that the model is able to predict classes quite well, but has problems with precise object localization.

This is an expected result, because the model uses a sliding window technique, which does not place the rectangle exactly on the coin, but only moves it by a fixed step (in our case 32 pixels), which causes the positions proposed by the model to be only in the vicinity of the true coins. Additionally, the size of the returned windows is fixed, which further worsens the mAP results for higher IoU threshold values.

Your task is to implement a model that will improve these results.

# Your Solution
In this section, you should place your solution. Make changes only here!

In [ ]:
# here you can implement your detector, which will return a list of detected objects
# in the format (x1, y1, x2, y2, label, confidence)


class YourDetector(nn.Module):
    def __init__(self):
        # here you can initialize your model
        super(YourDetector, self).__init__()
        pass

    def forward(self, img: torch.Tensor) -> list:
        """
        Your coin detection function on an image.

        Takes:
            img: Image to be processed.

        Returns:
            list of detected objects in the form of tuples
            (x1, y1, x2, y2, label, confidence).
        """
        # implement the logic of your detector here
        return []  # example model returning an empty list

In [ ]:
# definitions of augmentations for the training and validation sets.
# By default, None means no augmentation
train_transform = None
val_transform = None

train_ds, val_ds = setup_data(train_transform, val_transform, root="data/")

# train your model here
# ...

your_model = YourDetector()

# Evaluation

Running the cell below will allow you to check how many points your solution would score on the validation data. Before submitting, make sure that the entire notebook executes from start to finish without errors and without requiring user intervention after selecting the "Run All" option.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

if not FINAL_EVALUATION_MODE:
    your_out = predict_all_bounding_boxes(your_model, val_ds)

    your_map_val = calculate_map(
        your_out, val_ds, return_all=False
    )  # zwracamy tylko uśrednione mAP (główne kryterium oceny)

    score = (np.clip(your_map_val, 0.2, 0.85) - 0.2) / 0.65 * 100
    score = int(round(score))

    print(f"mAP na zbiorze walidacyjnym: {your_map_val:.2f}")
    print(f"Estymowana liczba punktów za zadanie: {score}")

    plot_detection_results_grid(your_out, val_ds)

    plot_confusion_matrix(your_out, val_ds, iou_threshold=0.5)
    plot_confusion_matrix(your_out, val_ds, iou_threshold=0.8)

During evaluation, the model will be saved as `your_model.pkl` and evaluated on the test set.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

if FINAL_EVALUATION_MODE:
    import cloudpickle

    # Gdy model posiada parametry, ustaw go w trybie ewaluacji i przenieś na CPU
    if list(your_model.parameters()):
        your_model.eval()
        your_model.cpu()

    OUTPUT_PATH = "file_output"
    FUNCTION_FILENAME = "your_model.pkl"
    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, FUNCTION_FILENAME)

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(your_model, f)